# Gráficas Test 1 ($T_n$)

Carga los CSV generados por `01_run_test1.ipynb` (o el script equivalente) y genera todas las gráficas en `results/figures/`. También permite mostrarlas inline para exploración.

Cambia `use_quick = True` para usar los datos quick si aún no has corrido el full.

In [ ]:
# --- Setup ---
import sys, os
from pathlib import Path

ROOT = Path.cwd().parent if (Path.cwd().name == 'notebooks') else Path.cwd()
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

from src.plotting import (
    plot_type_i_error, plot_power_curves, plot_runtime,
    plot_power_vs_cost, plot_pvalue_distribution_h0,
)

print(f'Working dir: {Path.cwd()}')

## Cargar CSV

In [ ]:
use_quick = False     # True: usa los CSVs '_quick'. False: usa los completos.

suffix = '_quick' if use_quick else ''
raw_csv = ROOT / 'results' / 'data' / f'tn_simulation_raw{suffix}.csv'
sum_csv = ROOT / 'results' / 'data' / f'tn_simulation_summary{suffix}.csv'

if not raw_csv.exists():
    raise FileNotFoundError(
        f'No se encontró {raw_csv}. Ejecuta primero `01_run_test1.ipynb`.'
    )

df = pd.read_csv(raw_csv)
summary = pd.read_csv(sum_csv)
print(f'Cargado: {raw_csv.name} ({len(df):,} filas)  /  {sum_csv.name} ({len(summary)} filas)')
print(f'Distribuciones: {sorted(df.dist.unique())}')
print(f'n: {sorted(df.n.unique())}, estimadores: {sorted(df.estimator.unique())}')

## Resumen pivotado

In [ ]:
print('Tasa de rechazo:')
display(summary.pivot_table(index=['dist','estimator'], columns='n', values='reject_rate').round(3))
print('\nTiempo medio por test (s):')
display(summary.pivot_table(index=['dist','estimator'], columns='n', values='mean_time_s').round(3))

## Generar todas las figuras

Se guardan en `results/figures/`. Cada celda muestra la figura inline.

In [ ]:
fig_dir = ROOT / 'results' / 'figures'
fig_dir.mkdir(parents=True, exist_ok=True)
alpha = 0.05

def show(path):
    if path is not None and Path(path).exists():
        display(Image(filename=str(path)))
    else:
        print(f'(sin imagen)')

### Error Tipo I bajo H₀

In [ ]:
show(plot_type_i_error(summary, alpha=alpha, outdir=fig_dir))

### Curvas de potencia bajo Hₐ

In [ ]:
show(plot_power_curves(summary, outdir=fig_dir))

### Tiempo de ejecución vs n

In [ ]:
show(plot_runtime(summary, outdir=fig_dir))

### Costo computacional vs potencia

In [ ]:
show(plot_power_vs_cost(summary, outdir=fig_dir))

### Distribución del p-valor bajo H₀ (sanity check: debería ser ~U(0,1))

In [ ]:
show(plot_pvalue_distribution_h0(df, outdir=fig_dir))

## Exploración interactiva

Plantilla para hacer tus propios cortes/gráficas.

In [ ]:
# Ejemplo: histograma del estadístico T_n para una celda específica
dist = 'Gamma(k=2.0,s=1.0)'
estimator = 'argmin'
n = 80

sub = df[(df['dist']==dist) & (df['estimator']==estimator) & (df['n']==n)]
fig, ax = plt.subplots(figsize=(6,3.5))
ax.hist(sub['statistic'], bins=30, color='C0', edgecolor='white')
ax.set_xlabel('T_n observado')
ax.set_ylabel('frecuencia')
ax.set_title(f'{dist} | {estimator} | n={n}')
ax.grid(True, alpha=0.3)
plt.show()